# AutoBI — Step 3: Profiling, Declarative Cleaning, Lineage, and EDA (SYNTHETIC)

This notebook demonstrates:
1. Profiling the **SYNTHETIC** sample datasets (`sales`, `HR`, `healthcare`) into a `ProfileReport`.
2. Running a declarative, allowlisted `CleaningPlan` (pure function, idempotent) with full column-level lineage tracking.
3. Generating Matplotlib/Seaborn EDA charts (distribution, correlation heatmap, time trend, category breakdown).

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "core").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from core.cleaning import execute_cleaning_plan
from core.lineage import LineageTracker
from core.profiling import (
    plot_category_breakdown,
    plot_correlation_heatmap,
    plot_distribution,
    plot_time_trend,
    profile_dataset,
)
from core.schemas import CleaningPlan

sales_df = pd.read_csv(PROJECT_ROOT / "sample_data" / "SYNTHETIC_sales.csv")
hr_emp_df = pd.read_excel(PROJECT_ROOT / "sample_data" / "SYNTHETIC_hr.xlsx", sheet_name="employees", header=2)
hr_dept_df = pd.read_excel(PROJECT_ROOT / "sample_data" / "SYNTHETIC_hr.xlsx", sheet_name="departments")
health_df = pd.read_csv(PROJECT_ROOT / "sample_data" / "SYNTHETIC_healthcare.csv")

tables = {
    "synthetic_sales": sales_df,
    "synthetic_hr_employees": hr_emp_df,
    "synthetic_hr_departments": hr_dept_df,
    "synthetic_healthcare": health_df,
}
list(tables.keys())

In [ ]:
report = profile_dataset(tables, batch_id="demo_synthetic")
for tbl in report.tables:
    print(f"=== {tbl.table_name} ({tbl.row_count} rows, keys={tbl.candidate_keys}) ===")
    for col in tbl.columns:
        print(f"  - {col.column_name}: {col.semantic_type} (nulls={col.null_percentage}%, distinct={col.distinct_count})")

print("\nCandidate relationships:")
for rel in report.candidate_relationships:
    print(f"  {rel.left_table}.{rel.left_column} -> {rel.right_table}.{rel.right_column} "
          f"(conf={rel.confidence}, overlap={rel.value_overlap}, {rel.cardinality_hint})")

In [ ]:
plan = CleaningPlan(
    table_name="synthetic_sales",
    description="Standardize sales types and drop constant metadata column",
    operations=[
        {"op": "parse_date", "column": "order_date"},
        {"op": "trim_text", "column": "region"},
        {"op": "standardize_case", "column": "region", "case": "title"},
        {"op": "cast_type", "column": "quantity", "target_type": "int"},
        {"op": "drop_duplicates"},
        {"op": "drop_column", "column": "data_origin"},
    ],
)

cleaned_sales, exec_result = execute_cleaning_plan(sales_df, plan)
cleaned_sales_again, _ = execute_cleaning_plan(cleaned_sales, plan)
pd.testing.assert_frame_equal(cleaned_sales, cleaned_sales_again)

tracker = LineageTracker(exec_result.lineage_events)
graph = tracker.build_graph({"synthetic_sales": list(cleaned_sales.columns)})
print(f"Cleaning idempotent! Lineage events recorded: {len(graph.events)}")
for chain in graph.column_chains[:5]:
    print(f"  {chain.target_column} <- {chain.origin_columns} via {chain.operations}")

In [ ]:
import matplotlib.pyplot as plt

fig_dist = plot_distribution(cleaned_sales, "revenue", title="SYNTHETIC Sales Revenue Distribution")
fig_corr = plot_correlation_heatmap(cleaned_sales, title="SYNTHETIC Sales Numeric Correlations")
fig_trend = plot_time_trend(cleaned_sales, "order_date", "revenue", freq="ME", title="SYNTHETIC Monthly Revenue")
fig_cat = plot_category_breakdown(cleaned_sales, "product", "revenue", agg="sum", title="SYNTHETIC Revenue by Product")

print("Generated 4 EDA figures:", type(fig_dist).__name__, type(fig_corr).__name__, type(fig_trend).__name__, type(fig_cat).__name__)
for fig in (fig_dist, fig_corr, fig_trend, fig_cat):
    plt.close(fig)